## Setup and data loading

In [23]:
import pandas as pd
import re

pd.set_option('display.max_columns', None)

In [25]:
forward_ratings = pd.read_csv('data/forward_ratings.csv')
midfielder_ratings = pd.read_csv('data/midfielder_ratings.csv')
defender_ratings = pd.read_csv('data/defender_ratings.csv')

print(forward_ratings.shape, midfielder_ratings.shape, defender_ratings.shape)

(195, 98) (463, 100) (502, 101)


In [26]:
with open('data/raw/championship_passing_octoparse_raw.csv', encoding='utf-8') as f:
    content = f.read()
print(content[:1500])

Player_URL,Player,Player1,Player_URL2,Player3,Player4,Player5,Apps,Mins,Assists,KeyP,AvgP,PS,Crosses,LongB,ThrB,Rating
https://www.whoscored.com/players/402779/show/femi-azeez,	1,Femi Azeez,https://www.whoscored.com/teams/192/show/england-millwall,"Millwall, ",25,",  AM(LR)  ",33(2),2812	,7	,1.9	,17.7	,59.4	,2.2	,0.6	,-	,7.36
https://www.whoscored.com/players/130035/show/lloyd-jones,	2,Lloyd Jones,https://www.whoscored.com/teams/160/show/england-charlton,"Charlton, ",30,",  D(C)  ",45(1),3946	,1	,0.4	,36	,74.9	,-	,1.9	,-	,7.26
https://www.whoscored.com/players/415367/show/caleb-taylor,	3,Caleb Taylor,https://www.whoscored.com/teams/192/show/england-millwall,"Millwall, ",23,",  D(C)  ",24(4),2243	,2	,0.3	,33.2	,78.5	,-	,2	,-	,7.25
https://www.whoscored.com/players/138695/show/oliver-mcburnie,	4,Oliver McBurnie,https://www.whoscored.com/teams/214/show/england-hull,"Hull, ",30,",  AM(CL),FW  ",32(5),2914	,7	,1.1	,18	,63.2	,-	,0.5	,-	,7.23
https://www.whoscored.com/players/235842/show/jake

## Extracting WhoScored's own player ratings

In [27]:
def parse_whoscored_ratings(filepath, league_name):
    df_ws = pd.read_csv(filepath)
    
    for col in df_ws.columns:
        if df_ws[col].dtype == object:
            df_ws[col] = df_ws[col].astype(str).str.replace('\t', '', regex=False).str.strip()
    
    df_ws = df_ws.rename(columns={'Player1': 'Player_clean', 'Player3': 'Squad'})
    df_ws['Squad'] = df_ws['Squad'].str.replace(',', '', regex=False).str.strip()
    df_ws['Rating'] = pd.to_numeric(df_ws['Rating'], errors='coerce')
    
    df_ws['league'] = league_name
    return df_ws[['Player_clean', 'Squad', 'Rating', 'league']].rename(columns={'Player_clean': 'Player'})

championship_ws = parse_whoscored_ratings('data/raw/championship_passing_octoparse_raw.csv', 'Championship')
bundesliga_ws = parse_whoscored_ratings('data/raw/bundesliga_passing_octoparse_raw.csv', '2. Bundesliga')

whoscored_ratings = pd.concat([championship_ws, bundesliga_ws], ignore_index=True)
print(whoscored_ratings.shape)
print(whoscored_ratings.head(10))

(939, 4)
            Player        Squad  Rating        league
0       Femi Azeez     Millwall    7.36  Championship
1      Lloyd Jones     Charlton    7.26  Championship
2     Caleb Taylor     Millwall    7.25  Championship
3  Oliver McBurnie         Hull    7.23  Championship
4      Jake Cooper     Millwall    7.18  Championship
5      Léo Scienza  Southampton    7.17  Championship
6      Matt Clarke        Derby    7.16  Championship
7     Bobby Thomas     Coventry    7.16  Championship
8    Tristan Crama     Millwall    7.15  Championship
9    Jacob Greaves      Ipswich    7.15  Championship


## Combine own ratings and merge with WhoScored

In [28]:
import unicodedata

def normalize_name(name):
    if pd.isna(name):
        return name
    nfkd = unicodedata.normalize('NFKD', str(name))
    return ''.join(c for c in nfkd if not unicodedata.combining(c)).lower().strip()

your_ratings = pd.concat([
    forward_ratings[['Player', 'Squad', 'league', 'rating']],
    midfielder_ratings[['Player', 'Squad', 'league', 'rating']],
    defender_ratings[['Player', 'Squad', 'league', 'rating']],
], ignore_index=True)

your_ratings = your_ratings[your_ratings['league'].isin(['Championship', '2. Bundesliga'])].copy()

your_ratings['_join_player'] = your_ratings['Player'].apply(normalize_name)
whoscored_ratings['_join_player'] = whoscored_ratings['Player'].apply(normalize_name)

comparison = your_ratings.merge(
    whoscored_ratings[['_join_player', 'Rating']], on='_join_player', how='inner'
)

print(f"Matched {len(comparison)} players with both ratings")
print(comparison[['Player', 'Squad', 'league', 'rating', 'Rating']].head(10))

Matched 488 players with both ratings
               Player             Squad        league  rating  Rating
0    Patrick Agyemang      Derby County  Championship    53.7    6.77
1      Cameron Archer       Southampton  Championship    56.5    6.30
2      Adam Armstrong       Southampton  Championship    58.0    6.79
3  Sinclair Armstrong      Bristol City  Championship    47.8    6.25
4         Jordan Ayew    Leicester City  Championship    46.6    6.55
5           Ivan Azón      Ipswich Town  Championship    55.4    6.42
6     Patrick Bamford  Sheffield United  Championship    54.5    6.80
7        Colby Bishop        Portsmouth  Championship    39.5    6.63
8      Rumarn Burrell               QPR  Championship    55.6    6.58
9     Tyrese Campbell  Sheffield United  Championship    46.0    6.50


## Spearman correlation

In [29]:
from scipy.stats import spearmanr

correlation, p_value = spearmanr(comparison['rating'], comparison['Rating'])

print(f"Spearman correlation: {correlation:.3f}")
print(f"P-value: {p_value:.5f}")

Spearman correlation: 0.621
P-value: 0.00000


Correlates at ρ = 0.621 (p < 0.001) with WhoScored's ratings — sits comfortably inside the 0.42–0.77 range Kolbowicz et al. (2024) found for the same kind of comparison, so this isn't out of line with what's been published.

## Where do the two systems disagree most?

Correlation only tells you the two systems roughly agree, not where they disagree. This looks at the biggest gaps between my TOPSIS rating and WhoScored's — usually the most interesting players to look at.

In [30]:
comparison['rating_rank'] = comparison['rating'].rank(ascending=False)
comparison['whoscored_rank'] = comparison['Rating'].rank(ascending=False)
comparison['rank_difference'] = comparison['whoscored_rank'] - comparison['rating_rank']

print("Players YOUR system rates much higher than WhoScored:")
print(comparison.sort_values('rank_difference', ascending=False)[
    ['Player', 'Squad', 'league', 'rating', 'Rating', 'rank_difference']
].head(10).to_string(index=False))

print("\nPlayers WhoScored rates much higher than YOUR system:")
print(comparison.sort_values('rank_difference', ascending=True)[
    ['Player', 'Squad', 'league', 'rating', 'Rating', 'rank_difference']
].head(10).to_string(index=False))

Players YOUR system rates much higher than WhoScored:
              Player             Squad        league  rating  Rating  rank_difference
      Cameron Archer       Southampton  Championship    56.5    6.30            407.0
           Ivan Azón      Ipswich Town  Championship    55.4    6.42            341.0
       Jonas Sterner           Dresden 2. Bundesliga    51.7    5.97            337.5
Mathias Kvistgaarden      Norwich City  Championship    53.0    6.39            315.5
      Stefano Marino      Paderborn 07 2. Bundesliga    57.4    6.52            297.0
  Christoph Daferner           Dresden 2. Bundesliga    53.0    6.43            291.0
           Sam Smith           Wrexham  Championship    54.0    6.46            285.0
         Ellis Simms     Coventry City  Championship    59.3    6.59            281.0
        George Hirst      Ipswich Town  Championship    57.2    6.55            277.5
      Charlie Kelman Charlton Athletic  Championship    50.6    6.38            260.0


In [34]:
players_df = pd.read_csv('data/all_players.csv')
position_lookup = players_df[['Player', 'Squad', 'Pos']].drop_duplicates()

comparison_with_pos = comparison.merge(position_lookup, on=['Player', 'Squad'], how='left')

print(comparison_with_pos.groupby('Pos')['rank_difference'].agg(['mean', 'count']).sort_values('mean', ascending=False))

            mean  count
Pos                    
FW     90.881356     59
FW,MF  87.820000     25
DF,MF  37.121951     41
DF,FW  23.750000      2
DF     -9.303797    158
MF    -37.719212    203


Averaged across the whole matched dataset (not just a few examples), there's a clean gradient by position: forwards +91 average rank vs TOPSIS, midfielders −38 (WhoScored rates them higher), defenders roughly neutral. Fits a theory: WhoScored's per-match average rewards all-round involvement, which favours midfielders since they touch the ball constantly. My season-total approach rewards cumulative output, which favours forwards since their value is concentrated in goal-scoring moments rather than spread across 90 minutes.

In [35]:
comparison_with_pos.to_csv('data/rating_validation_comparison.csv', index=False)
print(f"Saved {len(comparison_with_pos)} comparison rows to data/rating_validation_comparison.csv")

Saved 488 comparison rows to data/rating_validation_comparison.csv
